# Open Notebook in Colab





[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/childmindresearch/llm_tracker/blob/main/camilla_tutorial_v2.ipynb)

Please cite this if you use llm_tracker: <br>
Low, D.M., Mair, P., Nock, M., & Ghosh, S. (2026). Text Psychometrics: Assessing Psychological Constructs in Text Using Natural Language Processing. PLOS Digital Health.


# LLM Tracker: a researcher’s guide

Find psychological constructs in text, then compare the LLM’s coding with human coding.

**How to use this notebook:** run cells from top to bottom with the play button or **Shift+Enter**. Only edit the settings marked below. If a cell shows an error, stop and resolve it before continuing.

**Before you start**
- Use only data approved for sharing with your model provider. Remove identifying information and check your institution’s requirements and the provider’s data-retention policy. See [here](https://github.com/childmindresearch/llm_tracker/blob/main/tutorials/anonymization_tutorial.ipynb) for a guide to anonymization.
- You need an [OpenRouter API key](https://openrouter.ai/). LLM coding and quote matching cost money; test on a few documents first. Never share your key or save it in this notebook.
- LLM coding benefits from comparison with human codes (see section 3). Agreement with human coding is evidence of performance on your dataset, not proof of clinical validity.

# About Construct Detection

**LLM Tracker** is a Python package for identifying instances of psychological
constructs in qualitative text data using large language models.

A psychological construct is a concept describing an internal psychological state, trait, process, capacity, experience, or pattern of relating that can be inferred from what a person says, feels, thinks, perceives, or does. In the following example, NLP is used to detect the constructs of loneliness and anxiety in a sentence:


<img src="construct_demo.png" alt="Construct demo" width="400">

## Workflow

1. **Input files** — provide your documents and construct codebook.
2. **Run LLM** — identify constructs in your documents.
3. **Load human coding** — add human-coded data. Data types (long, wide) are described in Step 3.
4. **Compute reliability** — evaluate agreement with human/ground truth codes.
5. **Compare models** — summarize results from multiple models in one table.

<img src="workflow.png" alt="workflow" width="300">

## Setup (once)

Use Python 3.11–3.13. Open this notebook from a downloaded copy of the LLM Tracker repository. Run the next cell to install the package, then **restart the notebook kernel** if prompted. No optional tutorial packages are needed.

In [1]:
from pathlib import Path

project_dir = Path.cwd()
if not (project_dir / "pyproject.toml").exists():
    project_dir = project_dir.parent
if not (project_dir / "pyproject.toml").exists():
    raise FileNotFoundError("Open this notebook from the repository or its tutorials folder.")

%pip install -e "{project_dir}"


Obtaining file:///Users/camilla.strauss/Desktop/llm_tracker
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llm-tracker (pyproject.toml) ... done
  Created wheel for llm-tracker: filename=llm_tracker-0.1.0b1-py3-none-any.whl size=5085 sha256=f296dab780fb39d13953d8e4a5726e97cdb1ef3081eae0ce0f3c9fff5533a931
  Stored in directory: /private/var/folders/8t/3zxw9ymj0dz7gvv5vyvt45j4wh1rnl/T/pip-ephem-wheel-cache-vunn7xz6/wheels/bd/8a/b9/91e400720409b822150ba8c66230edd4496a7f13b4a347b60c
Successfully built llm-tracker
  Attempting uninstall: llm-tracker
    Found existing installation: llm-tracker 0.1.0b1
    Uninstalling llm-tracker-0.1.0b1:
      Successfully uninstalled llm-tracker-0.1.0b1

[notice] A new release of pip is available: 24.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you m

## 1. Input files

Choose for files:
- **Text:** a CSV with one document per row, or a folder of `.txt` files.
- **Codebook:** a JSON file listing construct names and definitions (see [example](../codebooks/suicide_risk_codebook.json)). You can ask an LLM/chatbot to convert your codebook into JSON.
- **Human coding** (for comparison): coded excerpts or presence/absence ratings for the same documents.

In [2]:
from getpass import getpass
from pathlib import Path

from IPython.display import display

from llm_tracker import AnalyzerConfig, LLMTrackerAnalyzer
from llm_tracker.comparison import (
    LLMTrackerComparer,
    compute_binary_metrics,
    compute_summary_tables,
    format_concatenated,
)
from llm_tracker.file_handlers import (
    codebook_constructs,
    load_codebook,
    load_human_coding,
    load_human_wide,
    validate_constructs,
)
from llm_tracker.presence import compare_presence
from llm_tracker.utils import format_coding_table


/Users/camilla.strauss/Desktop/llm_tracker/src/llm_tracker/models.py:38: UserWarning: Field name "construct" in "ConstructInstance" shadows an attribute in parent "BaseModel"
  class ConstructInstance(BaseModel):


Run the next cell and paste your API key when asked; it will be hidden.

In [7]:
api_key = getpass("OpenRouter API key: ").strip()
if not api_key:
    raise ValueError("Enter your OpenRouter API key.")

The settings cell defaults to the bundled practice sample in `sample_data/tutorial_practice/`, so you can run every section without your own data. For your own study, replace the example paths and column names with yours. Use full file paths. For CSV input, document IDs should be unique and match your human coding; `None` uses row numbers starting at 0. For `.txt` files, IDs are filenames without `.txt`.


Compare latest models and costs at [openrouter.ai/models](openrouter.ai/models). Copy your model's ID and paste it after `llm_model`.

In [8]:
# EDIT these settings.
# The defaults point at the bundled practice sample so you can run the whole
# tutorial end to end. Replace them with your own study files for real use.
sample_dir = project_dir / "sample_data" / "tutorial_practice"
input_data = str(sample_dir / "documents.csv")  # Or a folder of .txt files.
codebook_path = str(sample_dir / "codebook.json")
text_column = "text"  # CSV column containing the document text.
id_column = "document_id"  # CSV document ID column, or None for row numbers.
llm_model = "~google/gemini-flash-latest" # Change to your prefered model ID: ie "google/gemini-3.8-flash" (notice no ~)

output_folder = "research_results"  # Change this for each new study/run.

## 2. Run LLM coding

This sends each document and your codebook to the model. Results are saved automatically in a dated folder inside `output_folder`.

Review the extracted quotes below. **Do not continue to comparison if many documents failed.** The dated folder’s `errors` subfolder records failures; resolve them before rerunning. Rerunning this cell processes all documents again and incurs new charges.

In [9]:
source_path = Path(input_data)
if not source_path.exists():
    raise FileNotFoundError(f"Check input_data: {source_path}")
codebook = load_codebook(codebook_path)
config = AnalyzerConfig(api_key=api_key, model_name=llm_model, temperature=0)
analyzer = LLMTrackerAnalyzer(config=config)
coding_folder = str(Path(output_folder) / "LLM_coding")

if source_path.is_dir():
    results_llm, metadata_llm, errors_llm = analyzer.analyze_directory(
        input_dir=input_data,
        codebook_path=codebook_path,
        output_dir=coding_folder,
    )
else:
    results_llm, metadata_llm, errors_llm = analyzer.analyze_csv(
        csv_path=input_data,
        text_column=text_column,
        id_column=id_column,
        codebook_path=codebook_path,
        output_dir=coding_folder,
    )

if errors_llm:
    display(errors_llm)
    raise RuntimeError("Some documents failed. Resolve the errors before comparison.")
if not results_llm:
    raise ValueError("No documents were coded. Check your input files.")

display(format_coding_table(results_llm))


Processing [1/4]: doc01
  OK: Found 3 construct instances
  Progress: Successful 1/4; Errors 0/4
Processing [2/4]: doc02
  OK: Found 1 construct instances
  Progress: Successful 2/4; Errors 0/4
Processing [3/4]: doc03
  OK: Found 1 construct instances
  Progress: Successful 3/4; Errors 0/4
Processing [4/4]: doc04
  OK: Found 1 construct instances
  Progress: Successful 4/4; Errors 0/4

Analysis complete!
Output directory: /Users/camilla.strauss/Desktop/llm_tracker/tutorials/research_results/LLM_coding_2026-10-30_114322
Successful: 4/4
Coding Table
------------
One row per construct instance across all documents. Each row represents
a single quote identified by the LLM as an instance of a construct.

Columns:
  doc_id      : document identifier (filename or subreddit_author)
  construct   : psychological construct the instance belongs to
  quote       : exact quote extracted from the source text
  speaker_id  : speaker identifier if available in the source text
  quote_index : character

,doc_id,construct,quote,speaker_id,quote_index,confidence
0,doc01,anxiety,I feel worried about my exams.,None,0:30,2
1,doc01,social_support,My sister listens when I need to talk,None,31:68,2
2,doc01,self_management,I take a walk each evening to manage my stress,None,74:120,2
3,doc02,anxiety,I worry every night and I cannot sleep because...,None,0:68,2
4,doc03,social_support,My friend helped me carry my groceries when I ...,None,0:82,2
5,doc04,self_management,I started journaling every morning to keep tra...,None,0:61,2


## 3. Load human coding (optional)

**If you only need LLM-coded quotes, stop here.** To compare your LLM results against human-coded data, choose one of the three options below.

| Set `human_source` to | Use this when… |
| --- | --- |
| `synthetic` | You have no human data and want to practice the workflow with the bundled sample. |
| `long` | Your file has one row per coded passage, such as a Dedoose export. |
| `wide` | Your file has one row per document and a column for each construct. |

Read the option that fits your data, then edit the shared settings and run the loading cell. You do not need to prepare all three options.

### Option A: Synthetic human coding (practice only)

Set `human_source = "synthetic"`. The loading cell reads a small reference file that ships with this tutorial (`sample_data/tutorial_practice/synthetic_human_coding.csv`), matched to the bundled practice documents and codebook. **No second LLM run and no human-coded file are needed, and it makes no extra API calls.**

We wrote this reference as synthetic example data. It is **not** real human coding and does not measure agreement with humans; it only gives you something to compare against while you learn the workflow. For any real analysis, load your own human coding as ground truth using **Option B** (long) or **Option C** (wide) below.

### Option B: Long-format human coding (one row per passage)

Set `human_source = "long"`. A document can appear on several rows because the human highlighted several passages:

| document_id | quote | construct |
| --- | --- | --- |
| doc01 | I feel worried about exams. | anxiety |
| doc01 | My sister helps me cope. | social_support |
| doc02 | I worry every night. | anxiety |

Set the ID, quote, and construct column names to match your file. For Dedoose exports these are often `Media Title`, `Excerpt Copy`, and `Codes Applied Combined`. CSV and TSV files are supported; Excel requires the appropriate pandas Excel reader.

- If one cell contains multiple constructs, set `code_separator` to the separator used in your file. Use `;` rather than commas to separate if construct names contain commas. For one construct per row, choose a separator that never occurs in the names.
- Character positions are optional: leave `human_range_column = None` if unavailable. Otherwise set the column name and choose `"dash"` for `858-1159` or `"colon"` for `858:1159`.
- Document IDs must match the LLM input IDs. A file of passages usually omits documents where no constructs were found. Missing documents are **not** automatically treated as confirmed absences; use complete wide-format coding if this applies.

### Option C: Wide-format human coding (one row per document)

Set `human_source = "wide"`. Each construct has its own column: **0 = absent, 1 = present**.

| document_id | text | anxiety_score | social_support_score |
| --- | --- | --- | --- |
| doc01 | I feel worried. My sister helps me cope. | 1 | 1 |
| doc02 | I worry every night. | 1 | 0 |
| doc03 | The library opens at nine. | 0 | 0 |

Set `human_id_column` and `human_text_column` to your column names. The loader removes `_score` from column names, so `anxiety_score` becomes `anxiety`. If your columns use another ending, change `construct_suffix`.

Include a column for **every codebook construct** and complete numeric ratings for every document, including all-zero rows. Blank cells mean missing coding, not absence. `include_empty=True` in the loading cell keeps the all-zero documents.

Wide data records whether a construct appeared, not which passage or how many times. Use **binary presence comparison**, not quote matching or instance-count agreement. Keep `human_presence_threshold = 1` for 0/1 coding; change it only if your study uses a different rating rule (for example, 0.5 for averaged human ratings).

### Choose your source and comparison

Edit only the settings for your selected source; unused file settings are ignored.

- **`quotes`** extracts instances of phrases that express the construct (we later use a paid LLM to match the extracted phrases from the LLM and human-highlighted phrases).
- **`binary_presence`** detects whether each construct appears at least once anywhere in each document (no LLM call needed to compare with human results). Limitation: an LLM finding only one of multiple instances found by a human of a construct gets perfect reliability.

In [10]:
human_source = "synthetic" # EDIT: choose "synthetic", "long", or "wide".
comparison_level = "quotes"  # "binary_presence" must be used for wide data.

# For the synthetic practice option (bundled example data, not real human coding):
synthetic_human_path = str(sample_dir / "synthetic_human_coding.csv")

# For real human data (long or wide):
human_path = "/path/to/human_coding.csv"
human_id_column = "document_id"

# For long format only:
human_quote_column = "quote"
human_construct_column = "construct"
code_separator = ";"
human_range_column = None
human_range_format = "dash"

# For wide format only:
human_text_column = "text"
construct_suffix = "_score"
human_presence_threshold = 1

In [11]:
if comparison_level not in {"quotes", "binary_presence"}:
    raise ValueError('Set comparison_level to "quotes" or "binary_presence".')
if errors_llm or not results_llm:
    raise RuntimeError("Complete LLM coding successfully before loading comparison data.")

if human_source == "synthetic":
    human_results = load_human_coding(
        synthetic_human_path,
        doc_id_col=human_id_column,
        quote_col=human_quote_column,
        construct_col=human_construct_column,
        range_col=None,
        construct_separator=code_separator,
    )
    human_errors = []
    print("PRACTICE ONLY: this reference coding is synthetic example data we wrote, not real human judgments.")
elif human_source == "long":
    human_results = load_human_coding(
        human_path,
        doc_id_col=human_id_column,
        quote_col=human_quote_column,
        construct_col=human_construct_column,
        range_col=human_range_column,
        range_format=human_range_format,
        construct_separator=code_separator,
    )
elif human_source == "wide":
    if comparison_level != "binary_presence":
        raise ValueError('For wide data, set comparison_level to "binary_presence".')
    human_results = load_human_wide(
        human_path,
        doc_id_col=human_id_column,
        text_col=human_text_column,
        construct_suffix=construct_suffix,
        presence_threshold=human_presence_threshold,
        include_empty=True,
    )
else:
    raise ValueError('Set human_source to "synthetic", "long", or "wide".')

missing_ids = sorted(set(results_llm) - set(human_results))
if missing_ids:
    raise ValueError(f"Missing human coding for document IDs: {missing_ids}. Check IDs and coding completeness.")

human_results = {doc_id: human_results[doc_id] for doc_id in results_llm}
print(f"Reference coding loaded for all {len(results_llm)} LLM-coded documents.")
display(format_coding_table(human_results))

PRACTICE ONLY: this reference coding is synthetic example data we wrote, not real human judgments.
Reference coding loaded for all 4 LLM-coded documents.
Coding Table
------------
One row per construct instance across all documents. Each row represents
a single quote identified by the LLM as an instance of a construct.

Columns:
  doc_id      : document identifier (filename or subreddit_author)
  construct   : psychological construct the instance belongs to
  quote       : exact quote extracted from the source text
  speaker_id  : speaker identifier if available in the source text
  quote_index : character-level start:end indices of the quote
  confidence  : LLM confidence score (1=indirect or unclear evidence,
                2=clear; missing for human codings)
Absent or negated constructs do not produce instance rows.



,doc_id,construct,quote,speaker_id,quote_index,confidence
0,doc01,anxiety,I feel worried about my exams.,None,None,None
1,doc01,social_support,My sister listens when I need to talk,None,None,None
2,doc01,self_management,I take a walk each evening to manage my stress,None,None,None
3,doc02,anxiety,I worry every night and I cannot sleep because...,None,None,None
4,doc03,social_support,My friend helped me carry my groceries when I ...,None,None,None
5,doc04,self_management,I started journaling every morning to keep tra...,None,None,None


### 3.1 Check and map construct names

Names must match the codebook **exactly**, including capitalization. Run the next cell to list codebook names and flag any unknown names in your human coding.

If names differ but mean the same construct, add a pair to `construct_mapping` below: `"name in human data": "name in codebook"`. Only map equivalent constructs. If names were accidentally split at commas, fix `code_separator` and rerun loading instead of mapping fragments.

In [12]:
from llm_tracker.file_handlers import apply_construct_mapping

print("Construct names in your codebook:")
for name in codebook_constructs(codebook):
    print(f"- {name}")
unknown = validate_constructs(human_results, codebook)
print("Names needing correction:", unknown)


Construct names in your codebook:
- anxiety
- social_support
- self_management
Names needing correction: []


In [13]:
# EDIT if needed; otherwise leave empty. Remove # from your mapping lines.
construct_mapping = {
    # "social anxiety sx": "Social Anxiety",
}

if construct_mapping:
    human_results = apply_construct_mapping(human_results, construct_mapping)
validate_constructs(human_results, codebook, strict=True)
print("Construct names match. You can continue to comparison.")


Construct names match. You can continue to comparison.


## 4. Compute reliability

The next cell runs your selected comparison, using the loaded human/ground truth coding as the reference. Only the LLM-coded documents are evaluated; extra documents in the human file are excluded.

We have included several reliability metrics but also show you how to roll your own in section 4.1 below. 


| Measure | Research question |
| --- | --- |
| Sensitivity (recall) | How much of the human coding did the LLM find? |
| Precision | How much of the LLM coding agreed with human coding? |
| F1 | How well does the LLM balance sensitivity and precision? |
| Jaccard | How much overlap is there between the two sets of detections? |

Higher values generally mean better agreement. Missing/undefined values are not low performing and need to be reviewed. If `comparison_level` = 'quote', comparisons assess coded phrases; if `comparison_level` = 'binary_presence', comparisons assess whole documents, so their scores answer different questions.

Inspect disagreements, not just summary scores. Presence tables also include agreement on absence; many absent constructs can make accuracy look high even when detection is poor.

### Which metrics each `comparison_level` reports

Both levels report **sensitivity**, **precision**, **F1**, and **Jaccard** (from TP/FP/FN), plus these extras:

**`quotes`** (span level, `format_concatenated.csv`):

| Column | Meaning |
| --- | --- |
| `tp`, `fp`, `fn` | Span-level match counts |
| `pr_auc` | Precision–recall summary using LLM confidence |
| `interviews_with_construct` | Documents containing each construct |

No `tn`: a construct neither coder quoted produces no span.

**`binary_presence`** (document level, `binary_metrics.csv`):

| Column | Meaning |
| --- | --- |
| `n_docs`, `human_present`, `llm_present` | Documents rated and marked present |
| `tp`, `fp`, `fn`, `tn` | Full confusion matrix |
| `pabak`, `cohens_kappa`, `kripp_alpha_nominal` | Chance-corrected agreement |

**Why kappa is binary-only:** kappa's chance term needs all four cells, including TN. The `quotes` level has no TN, so kappa, PABAK, and Krippendorff's alpha are omitted. Jaccard appears at both levels because it ignores TN: `TP / (TP + FP + FN)`.

In [14]:
if errors_llm or not results_llm:
    raise RuntimeError("Complete LLM coding successfully before comparing.")
if comparison_level not in {"quotes", "binary_presence"}:
    raise ValueError('Set comparison_level to "quotes" or "binary_presence".')
if human_source == "wide" and comparison_level != "binary_presence":
    raise ValueError('Wide data requires comparison_level = "binary_presence".')
if human_source == "synthetic" and human_errors:
    raise RuntimeError("Resolve synthetic coding errors before comparing.")
validate_constructs(human_results, codebook, strict=True)

reference_label = "synthetic" if human_source == "synthetic" else "human"
summary_folder = Path(output_folder) / reference_label / comparison_level
print(f"Reference source: {human_source}")
if comparison_level == "binary_presence":
    comparison_table = compare_presence(
        human_results=human_results,
        llm_results=results_llm,
        constructs=list(codebook_constructs(codebook)),
        document_ids=list(results_llm),
        output_dir=summary_folder,
    )
    summary = compute_binary_metrics(comparison_table, output_dir=summary_folder)
else:
    comparator = LLMTrackerComparer(config=config)
    comparison_table = comparator.compare_results(
        human_results=human_results,
        llm_results=results_llm,
        output_dir=str(summary_folder / "comparison"),
    )
    per_document, pooled, weighted = compute_summary_tables(
        comparison_table, output_dir=summary_folder,
    )
    summary = format_concatenated(pooled)
    summary.to_csv(summary_folder / "format_concatenated.csv", index=False)

display(summary)
display(comparison_table.head(20))
print(f"Results saved under: {summary_folder.resolve()}")


Reference source: synthetic


,construct,tp,fp,fn,sensitivity,precision,f1,jaccard,pr_auc,interviews_with_construct [p5-p95]
0,anxiety,2,0,0,1.00,1.00,1.00,1.00,-,2 [0.00-1.00]
1,self_management,2,0,0,1.00,1.00,1.00,1.00,-,2 [0.00-1.00]
2,social_support,2,0,0,1.00,1.00,1.00,1.00,-,2 [0.00-1.00]
3,Overall,6,0,0,1.00,1.00,1.00,1.00,-,4 [0.00-1.00]


,doc_id,construct,status,human_quote,llm_quote,human_indices,llm_indices,human_confidence,llm_confidence,paraphrase,span_overlap,match_confidence,tp,fp,fn
0,doc01,anxiety,matched,I feel worried about my exams.,I feel worried about my exams.,None,0:30,None,1,False,None,1.00,1,0,0
1,doc01,self_management,matched,I take a walk each evening to manage my stress,I take a walk each evening to manage my stress,None,74:120,None,1,False,None,1.00,1,0,0
2,doc01,social_support,matched,My sister listens when I need to talk,My sister listens when I need to talk,None,31:68,None,1,False,None,1.00,1,0,0
3,doc02,anxiety,matched,I worry every night and I cannot sleep because...,I worry every night and I cannot sleep because...,None,0:68,None,1,False,None,1.00,1,0,0
4,doc03,social_support,matched,My friend helped me carry my groceries when I ...,My friend helped me carry my groceries when I ...,None,0:82,None,1,False,None,0.98,1,0,0
5,doc04,self_management,matched,I started journaling every morning to keep tra...,I started journaling every morning to keep tra...,None,0:61,None,1,False,None,1.00,1,0,0


Results saved under: /Users/camilla.strauss/Desktop/llm_tracker/tutorials/research_results/synthetic/quotes


### 4.1 Roll your own metrics

LLM Tracker already implements several agreement metrics for you: precision, sensitivity, F1, Jaccard, PABAK, Cohen's kappa, and Krippendorff's alpha for detection, plus ordinal metrics on instance counts. When your study needs something the package does not ship, you can compute it yourself: everything the built-in metrics use is available in an ordinary pandas DataFrame. This subsection shows how, working through two examples: **accuracy** on binary presence/absence and **R²** on ordinal instance counts.

**Which table do you need?** This is the decision that matters most, and getting it wrong is the easiest way to compute a number that looks fine and means nothing.

A `quotes` `comparison_table` is **span-level**: one row per coded instance, labelled `matched`, `human_only`, or `llm_only`. It has no true negatives, because a construct neither coder marked produces no row at all. Any metric whose formula contains TN cannot be computed from it.

The **presence grid** is **document-level**: it crosses every document with every construct, so pairs neither coder marked appear as explicit `0 / 0` rows — the true negatives. When `comparison_level = "binary_presence"`, your `comparison_table` is already this grid. When `comparison_level = "quotes"`, derive it with `build_presence_grid(comparison_table)`. The next cell does whichever applies.

| Column | Meaning |
|---|---|
| `doc_id`, `construct` | The cell being rated |
| `human_count`, `llm_count` | How many instances each coder marked |
| `human_present`, `llm_present` | The counts binarized at the presence threshold |

Accuracy needs TN, so it reads the grid. R² on counts also needs the zero cells — a document where the LLM correctly found nothing is real information about its calibration — so it reads the grid too.

In [15]:
from llm_tracker.comparison import build_presence_grid

if comparison_level == "binary_presence":
    presence_grid = comparison_table  # compare_presence already returns the grid
else:
    presence_grid = build_presence_grid(comparison_table)

presence_grid.head(10)

,doc_id,construct,human_count,llm_count,human_present,llm_present
0,doc01,anxiety,1,1,1,1
1,doc01,self_management,1,1,1,1
2,doc01,social_support,1,1,1,1
3,doc02,anxiety,1,1,1,1
4,doc02,self_management,0,0,0,0
5,doc02,social_support,0,0,0,0
6,doc03,anxiety,0,0,0,0
7,doc03,self_management,0,0,0,0
8,doc03,social_support,1,1,1,1
9,doc04,anxiety,0,0,0,0


Accuracy is the proportion of document × construct cells where the two coders reach the same present/absent verdict: `(TP + TN) / total`. On the grid that is an elementwise comparison of the two presence columns.

R² uses the counts themselves instead of discarding them to presence, asking how much of the variance in the human's counts the LLM's counts account for: `1 - SS_residual / SS_total`, with the human counts as the reference.

Both helpers return `NaN` rather than a number where the statistic is undefined. This mirrors what the package does internally in `_safe_kappa` and `_safe_alpha`: a construct the human never coded has zero variance in its counts, and silently reporting `0.0` there would look like a real, poor score instead of an absent one.

In [16]:
import numpy as np
import pandas as pd


def accuracy(human_present, llm_present):
    """Proportion of (document, construct) cells where the coders agree."""
    if len(human_present) == 0:
        return np.nan
    return float((np.asarray(human_present) == np.asarray(llm_present)).mean())


def r2(human_counts, llm_counts):
    """R^2 of the LLM's instance counts against the human's."""
    human_counts = np.asarray(human_counts, dtype=float)
    llm_counts = np.asarray(llm_counts, dtype=float)
    if len(human_counts) < 2:
        return np.nan
    ss_total = ((human_counts - human_counts.mean()) ** 2).sum()
    if ss_total == 0:  # human never varied — nothing to explain
        return np.nan
    ss_residual = ((human_counts - llm_counts) ** 2).sum()
    return float(1 - ss_residual / ss_total)


def custom_metrics(grid):
    """Accuracy and R^2 per construct, plus a pooled Overall row."""
    groups = [(str(name), g) for name, g in grid.groupby("construct")]
    groups.append(("Overall", grid))

    rows = []
    for construct, g in groups:
        rows.append(
            {
                "construct": construct,
                "n_cells": len(g),
                "human_present": int(g["human_present"].sum()),
                "llm_present": int(g["llm_present"].sum()),
                "accuracy": round(
                    accuracy(g["human_present"], g["llm_present"]), 4
                ),
                "r2_counts": round(r2(g["human_count"], g["llm_count"]), 4),
            }
        )
    return pd.DataFrame(rows)


custom_metrics(presence_grid)

,construct,n_cells,human_present,llm_present,accuracy,r2_counts
0,anxiety,4,2,2,1.0,1.0
1,self_management,4,2,2,1.0,1.0
2,social_support,4,2,2,1.0,1.0
3,Overall,12,6,6,1.0,1.0


## 5. Compare results from different models (optional)

Run each model on the **same documents, human coding, codebook, and comparison settings** first. Then enter the full paths to their `format_concatenated.csv` files below, with a label for each model.

This section reads those files only: it makes **no API calls** and can run without rerunning the earlier steps.

The displayed and saved table contains **one row per model**, summarizing all construct rows except the pre-existing `Overall` row.

Scores are summarized **across constructs**, not across documents: `mean (SD) [p5, p25, p50, p75, p95]; n=valid scores`. Each construct has equal weight. These are descriptive percentiles, not confidence intervals, and are different from pooled performance. Counts show `total;` followed by that distribution. Undefined scores (`-` or blank in the source) are excluded, not treated as zero; `n` shows how many remain. SD is unavailable for a single value.

The table also lists constructs with Jaccard **above 0.4**, at/below 0.4, or undefined. You can change the cutoff below. Source CSVs are never modified; the comparison CSV is saved separately.

In [17]:
# EDIT: add one path and one matching model label for each run.
summary_csv_paths = [
    "/path/to/model_A/format_concatenated.csv",
    "/path/to/model_B/format_concatenated.csv",
]
model_labels = ["Model A", "Model B"]

jaccard_cutoff = 0.4
comparison_csv_path = "model_summary_comparison.csv"


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def describe_values(values):
    values = values.dropna()
    if values.empty:
        return "-; n=0"
    percentiles = ", ".join(f"{value:.2f}" for value in values.quantile([0.05, 0.25, 0.5, 0.75, 0.95]))
    sd = f"{values.std():.2f}" if len(values) > 1 else "-"
    return f"{values.mean():.2f} ({sd}) [{percentiles}]; n={len(values)}"


def summarize_model(frame, model_label, cutoff):
    row = {
        "model": model_label,
        "n_constructs": len(frame),
        "constructs_included": "; ".join(frame["construct"]),
    }
    for column in ["tp", "fp", "fn", "count"]:
        total = f"{frame[column].sum():.0f}" if not frame.empty else "0"
        row[column] = f"{total}; {describe_values(frame[column])}"
    for column in ["sensitivity", "precision", "f1", "jaccard", "pr_auc"]:
        row[column] = describe_values(frame[column])
    if "interviews_with_construct [p5-p95]" in frame:
        row["documents_per_construct"] = describe_values(frame["document_count"])
    above = frame.loc[frame["jaccard"] > cutoff, "construct"]
    below = frame.loc[frame["jaccard"] <= cutoff, "construct"]
    undefined = frame.loc[frame["jaccard"].isna(), "construct"]
    row["jaccard_cutoff"] = cutoff
    row["n_above_cutoff"] = len(above)
    row["constructs_above_cutoff"] = "; ".join(above)
    row["n_at_or_below_cutoff"] = len(below)
    row["constructs_at_or_below_cutoff"] = "; ".join(below)
    row["n_undefined_jaccard"] = len(undefined)
    row["constructs_undefined_jaccard"] = "; ".join(undefined)
    return row


if not summary_csv_paths or len(summary_csv_paths) != len(model_labels):
    raise ValueError("Enter at least one CSV path and a matching label for each path.")
if any(not label.strip() for label in model_labels) or len(set(model_labels)) != len(model_labels):
    raise ValueError("Model labels must be non-empty and unique.")
if not 0 <= jaccard_cutoff <= 1:
    raise ValueError("jaccard_cutoff must be between 0 and 1.")

input_paths = [Path(path).expanduser().resolve() for path in summary_csv_paths]
destination = Path(comparison_csv_path).expanduser().resolve()
if destination in input_paths:
    raise ValueError("The output path must not overwrite a source CSV.")

comparison_rows = []
required_columns = ["construct", "tp", "fp", "fn", "sensitivity", "precision", "f1", "jaccard", "pr_auc"]
for model_label, path in zip(model_labels, input_paths, strict=True):
    frame = pd.read_csv(path)
    missing = sorted(set(required_columns) - set(frame.columns))
    if missing:
        raise ValueError(f"{path}: missing columns {missing}. Use a default format_concatenated() CSV.")
    frame = frame[required_columns + [column for column in ["interviews_with_construct [p5-p95]"] if column in frame]].copy()
    frame = frame.loc[frame["construct"] != "Overall"].copy()
    if frame.empty or frame["construct"].isna().any() or frame["construct"].duplicated().any():
        raise ValueError(f"{path}: expected non-empty, unique construct rows.")
    frame["construct"] = frame["construct"].astype(str)
    for column in required_columns[1:]:
        frame[column] = pd.to_numeric(frame[column].replace("-", np.nan), errors="raise")
        values = frame[column].dropna()
        if not np.isfinite(values).all():
            raise ValueError(f"{path}: {column} contains infinite values.")
        if column in {"tp", "fp", "fn"}:
            if frame[column].isna().any() or (values < 0).any() or (values % 1 != 0).any():
                raise ValueError(f"{path}: {column} must contain non-negative whole counts.")
        elif not values.between(0, 1).all():
            raise ValueError(f"{path}: {column} must be between 0 and 1, or undefined.")
    if "interviews_with_construct [p5-p95]" in frame:
        document_counts = frame["interviews_with_construct [p5-p95]"].astype(str).str.extract(r"^(\d+)\s*\[")[0]
        if document_counts.isna().any():
            raise ValueError(f"{path}: invalid formatted document counts.")
        frame["document_count"] = pd.to_numeric(document_counts, errors="raise")
    frame["count"] = frame[["tp", "fp", "fn"]].sum(axis=1)
    row = summarize_model(frame, model_label, jaccard_cutoff)
    row["source_csv"] = str(path)
    comparison_rows.append(row)

model_comparison = pd.DataFrame(comparison_rows)
display(model_comparison)
destination.parent.mkdir(parents=True, exist_ok=True)
model_comparison.to_csv(destination, index=False)
print(f"Saved model comparison to: {destination}")
